In [1]:
# Langchain import
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings

# System import
from datetime import datetime, date
import sys
sys.path.insert(0, "..")

# Local import
from utils.data_processing import build_chroma_document_from_mongo_document
from utils.mongo_handler import get_legislation_by_query

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Enviroment import 
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path=".env")
# Load environment variables
MONGO_URI = os.getenv("MONGO_URI")
CHECKPOINT_DB = os.getenv("CHECKPOINT_DB")

LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING")
LANGSMITH_ENDPOINT = os.getenv("LANGSMITH_ENDPOINT")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT")

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
RERANK_MODEL = os.getenv("RERANK_MODEL")
# RERANK_MAX_LENGTH = int(os.getenv("RERANK_MAX_LENGTH"))

VECTOR_STORE_COLLECTION = os.getenv("VECTOR_STORE_COLLECTION")
VECTOR_STORE_HOST = os.getenv("VECTOR_STORE_HOST")
VECTOR_STORE_PORT = int(os.getenv("VECTOR_STORE_PORT"))

In [3]:
query = {"category": "Quyết định"}
legislations = get_legislation_by_query(query)
chroma_documents = [build_chroma_document_from_mongo_document(doc) for doc in legislations["data"]]
print(f"Number of documents: {len(chroma_documents)}")
print(f"First document: {chroma_documents[0]['documents'][:500]}...")
print("-" * 50)
print(f"First document metadata: {chroma_documents[0]['metadata']}")

Number of documents: 9079
First document: THỦ TƯỚNG CHÍNH PHỦ CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc Số: 871/QĐ-TTg Hà Nội, ngày 19 tháng 8 năm 2024 QUYẾT ĐỊNH BAN HÀNH KẾ HOẠCH TRIỂN KHAI THI HÀNH LUẬT QUẢN LÝ, SỬ DỤNG VŨ KHÍ, VẬT LIỆU NỔ VÀ CÔNG CỤ HỖ TRỢ THỦ TƯỚNG CHÍNH PHỦ Căn cứ Luật Tổ chức Chính phủ ngày 19 tháng 6 năm 2015; Luật sửa đổi, bổ sung một số điều của Luật Tổ chức Chính phủ và Luật Tổ chức chính quyền địa phương ngày 22 tháng 11 năm 2019; Căn cứ Luật Quản lý, sử dụng vũ khí, vật liệu nổ và công ...
--------------------------------------------------
First document metadata: {'id': '6825b5fa5733d64bd86fb9bf', 'name': 'QUYẾT ĐỊNH BAN HÀNH KẾ HOẠCH TRIỂN KHAI THI HÀNH LUẬT QUẢN LÝ, SỬ DỤNG VŨ KHÍ, VẬT LIỆU NỔ VÀ CÔNG CỤ HỖ TRỢ', 'category': 'Quyết định', 'department': 'Thủ tướng Chính phủ', 'numberDoc': '871/QĐ-TTg', 'fields': ['Bộ máy hành chính'], 'dateApproved': datetime.datetime(2024, 8, 19, 0, 0), 'createdAt': datetime.datetime(2025, 5, 15, 9,

In [4]:
# Ollama model initialization
llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL)
print(f"LLM model loaded: {llm.model}")

# Embedding model
# embedding_model = OllamaEmbeddings(model=model_name)
# print(f"Embedding model loaded: {embedding_model.model}")
embedding_model = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
print(f"Embedding model loaded: {embedding_model.model}")

# Vector store initialization
vector_store = Chroma(
    # persist_directory=persist_directory,
    host=VECTOR_STORE_HOST,
    port=VECTOR_STORE_PORT,
    collection_name=VECTOR_STORE_COLLECTION,
    embedding_function=embedding_model,
)
print(f"Total number of documents in the vector store: {vector_store._collection.count()}")

LLM model loaded: llama3.1
Embedding model loaded: bge-m3
Total number of documents in the vector store: 4208


In [5]:
# vector_store.delete_collection()

In [5]:
# Text splitter configuration
chunk_size = 3000  # chunk size (characters)
chunk_overlap = 300  # chunk overlap (characters)
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=chunk_size,  
#     chunk_overlap=chunk_overlap,  
#     add_start_index=True,  # track index in original document
# )   
text_splitter = SemanticChunker(
    embedding_model, breakpoint_threshold_type="percentile", add_start_index=True
)

# Sanitizing metadata function
def sanitize_metadata(metadata):
    """Sanitize metadata by removing keys that are not serializable."""
    for k, v in metadata.items():
        if isinstance(v, (dict, list)):
            metadata[k] = ", ".join(map(str, v))  # Convert to string
        if isinstance(v, (datetime, date)):
            metadata[k] = v.strftime("%Y-%m-%dT%H:%M:%S")  # Convert to ISO format string
    return metadata

from langchain_core.documents import Document

# Convert the documents to Chroma Document format
docs = [Document(page_content=doc["documents"], metadata=sanitize_metadata(doc["metadata"])) for doc in chroma_documents]

In [6]:
import time

for doc in docs:
    chunked_docs = text_splitter.split_documents([doc])
    print(f"Number of chunks {doc.metadata['name']} splited into: {len(chunked_docs)}")
    
    for attempt in range(3):  # Try up to 3 times
        try:
            # Operation that might fail
            vector_store.add_documents(
                documents=chunked_docs,
                collection_name="legislation",
                ids=[(str(doc.metadata["id"]) + "_" + str(doc.metadata["start_index"])) for doc in chunked_docs],
            )
            break  # Exit the loop on success
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}. Retrying...")
            time.sleep(2)  # Wait before retrying
    else:
        print("Failed to add documents after 3 attempts. Skipping this document.")

Number of chunks QUYẾT ĐỊNH BAN HÀNH KẾ HOẠCH TRIỂN KHAI THI HÀNH LUẬT QUẢN LÝ, SỬ DỤNG VŨ KHÍ, VẬT LIỆU NỔ VÀ CÔNG CỤ HỖ TRỢ splited into: 2
Number of chunks QUYẾT ĐỊNH BAN HÀNH QUY TRÌNH TIẾP NHẬN, GIẢI QUYẾT HỒ SƠ HAI NHÓM THỦ TỤC HÀNH CHÍNH LIÊN THÔNG ĐIỆN TỬ: ĐĂNG KÝ KHAI SINH, ĐĂNG KÝ THƯỜNG TRÚ, CẤP THẺ BẢO HIỂM Y TẾ CHO TRẺ EM DƯỚI 6 TUỔI; ĐĂNG KÝ KHAI TỬ, XÓA ĐĂNG KÝ THƯỜNG TRÚ, GIẢI QUYẾT MAI TÁNG PHÍ, TỬ TUẤT splited into: 2
Number of chunks QUYẾT ĐỊNH VỀ VIỆC CÔNG BỐ DANH MỤC THỦ TỤC HÀNH CHÍNH SỬA ĐỔI, BỔ SUNG MỨC PHÍ THEO THÔNG TƯ SỐ 43/2024/TT-BTC  NGÀY 28/6/2024 CỦA BỘ TÀI CHÍNH THUỘC PHẠM VI QUẢN LÝ CỦA BỘ Y TẾ splited into: 2
Number of chunks QUYẾT ĐỊNH BAN HÀNH DANH MỤC MÃ HÃNG SẢN XUẤT VẬT TƯ Y TẾ PHỤC VỤ QUẢN LÝ VÀ GIÁM ĐỊNH, THANH TOÁN CHI PHÍ KHÁM BỆNH, CHỮA BỆNH BẢO HIỂM Y TẾ (ĐỢT 12) splited into: 2
Number of chunks QUYẾT ĐỊNH PHÊ DUYỆT ĐỀ ÁN THÍ ĐIỂM XÂY DỰNG CỬA KHẨU THÔNG MINH TẠI ĐƯỜNG CHUYÊN DỤNG VẬN CHUYỂN HÀNG HÓA KHU VỰC MỐC 1119 -1120 VÀ ĐƯỜNG CHUYÊN D

In [7]:
query = "Thủ tục tố tụng hình sự gồm các bước nào"

In [8]:
retrieve_result = vector_store.similarity_search(
    query=query,
    k=10
)

for i, doc in enumerate(retrieve_result):
    print(f"Document {i + 1}:")
    print("Name:", doc.metadata.get("name", "N/A"))
    print("Content:", doc.page_content)
    print("Id:", doc.metadata.get("id", "N/A"))
    print("Document number:", doc.metadata.get("numberDoc", "N/A"))
    print("Fields:", doc.metadata.get("fields", "N/A"))
    print("Effective date:", doc.metadata.get("dateApproved", "N/A"))
    # print("Metadata:", retrieve_result["metadatas"][0][i])
    print("-"* 50)  # Separator for readability

Document 1:
Name: LUẬT TỔ CHỨC CƠ QUAN ĐIỀU TRA HÌNH SỰ
Content: Cơ quan, tổ chức, cá nhân có liên quan. Điều 3. Nguyên tắc tổ chức Điều tra hình sự1. Tuân thủ Hiến pháp và pháp luật. 2. Bảo đảm sự chỉ đạo, chỉ huy tập trung thống nhất, hiệu lực, hiệu quả; phân công, phân cấp rành mạch, chuyên sâu, tránh chồng chéo và được kiểm soát chặt chẽ; Điều tra kịp thời, nhanh chóng, chính xác, khách quan, toàn diện, đầy đủ, không để lọt tội phạm và không làm oan người vô tội. 3. Cơ quan Điều tra cấp dưới chịu sự hướng dẫn, chỉ đạo nghiệp vụ của Cơ quan Điều tra cấp trên; cá nhân chịu trách nhiệm trước cấp trên và trước pháp luật về hành vi, quyết định của mình. 4. Chỉ cơ quan, người có thẩm quyền quy định trong Luật này mới được tiến hành hoạt động điều tra hình sự. Điều 4.
Id: 688c42caffb358067b47fbde
Document number: 99/2015/QH13
Fields: Trách nhiệm hình sự, Thủ tục Tố tụng
Effective date: 2015-11-26T00:00:00
--------------------------------------------------
Document 2:
Name: LUẬT